# Madhava QR-JL (8D->64D): Benchmark vs HNSW / IVF / PQ

**Com Error Backpropagation Modulation**

Benchmark do Madhava QR-JL contra FAISS FlatIP, HNSW, IVF e PQ.

### Metodos:
| # | Metodo | Descricao |
|---|--------|-----------|
| A | FAISS FlatIP | Busca exata (ground truth) |
| B | HNSW(ef=32/64/128) | Graph-based ANN |
| C | IVF(nprobe=1/10/20) | Inverted File Index |
| D | PQ(m=8/16) | Product Quantization |
| **E** | **Madhava QR-JL** | **QR-JL + error backprop** |

**Ref:** Zenodo 20964487 | **Licenca:** BSL 1.1


In [ ]:
import sys,os,warnings,math,time,random,gc,json,copy
import numpy as np
os.environ['TOKENIZERS_PARALLELISM']='false'; warnings.filterwarnings('ignore')
for pkg,imp in [('faiss-cpu','faiss'),('scikit-learn','sklearn'),('scipy','scipy')]:
    try: __import__(imp)
    except: import subprocess; subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import faiss
SEED=42; np.random.seed(SEED); random.seed(SEED)
FINAL_K=10
print('Setup OK')


In [ ]:
class MadhavaQRJL:
    def __init__(self,seed=SEED):
        self.dims=[8,64]; self.full_dim=128
        self.keep_ratio=0.20; self.max_stage1=100000; self.final_topk=200
        self.rng=np.random.RandomState(seed+1)
        self.vectors=None; self.proj_L={}; self.error={}; self.proj_matrices={}
    def _make_orthogonal_proj(self,d_out,d_in):
        Q,_=np.linalg.qr(self.rng.randn(d_out,d_in).astype(np.float64).T)
        P=Q[:,:d_out].T.astype(np.float32)
        assert np.abs(P@P.T-np.eye(d_out)).max()<1e-5
        return P
    def build(self,vectors):
        t0=time.time(); self.vectors=vectors; self.n_vectors=len(vectors)
        self.norms=np.linalg.norm(vectors,axis=1).astype(np.float64)
        for d in self.dims:
            P=self._make_orthogonal_proj(d,self.full_dim)
            self.proj_matrices[d]=P
            proj=(vectors.astype(np.float32)@P.T).astype(np.float64)
            self.proj_L[d]=proj
            captured=np.linalg.norm(proj,axis=1)
            self.error[d]=np.sqrt(np.maximum(self.norms**2-captured**2,0))
        self.build_time=time.time()-t0; return self
    def _upper_bound(self,pv,ev,pq,eq):
        return pv@pq+ev*eq
    def _modulation_alpha(self,e1,e2):
        return 1.0/(1.0+np.exp(-(e1-e2)/max(np.mean(e1),1e-9)*0.5))
    def search(self,q,retprof=False):
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q)
        p={"n_total":self.n_vectors}
        P1=self.proj_matrices[8]
        q1=(q.astype(np.float32)@P1.T).astype(np.float64)
        qr1=math.sqrt(max(0,qn**2-np.linalg.norm(q1)**2))
        B1=self._upper_bound(self.proj_L[8],self.error[8],q1,qr1)
        k1=min(max(int(self.n_vectors*self.keep_ratio),2000),self.max_stage1,self.n_vectors)
        if self.n_vectors<=k1: idx1=np.arange(self.n_vectors)
        else: idx1=np.argpartition(-B1,k1-1)[:k1]
        p["n1"]=len(idx1)
        P2=self.proj_matrices[64]
        q2=(q.astype(np.float32)@P2.T).astype(np.float64)
        qr2=math.sqrt(max(0,qn**2-np.linalg.norm(q2)**2))
        B2=self._upper_bound(self.proj_L[64][idx1],self.error[64][idx1],q2,qr2)
        al=self._modulation_alpha(self.error[8][idx1],self.error[64][idx1])
        p["alpha"]=float(np.mean(al))
        mod=B1[idx1]+al*(B2-B1[idx1])
        k2=min(self.final_topk,len(idx1))
        idx2=idx1[np.argpartition(-mod,k2-1)[:k2]]
        top=idx2[np.argsort(-(self.vectors[idx2].astype(np.float32)@q.astype(np.float32)))[:FINAL_K]]
        if retprof: return top,p
        return top
    def bound_violations(self,q):
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q)
        tc=self.vectors.astype(np.float64)@q
        v={}
        for d in self.dims:
            P=self.proj_matrices[d]
            qd=(q.astype(np.float32)@P.T).astype(np.float64)
            ub=self._upper_bound(self.proj_L[d],self.error[d],qd,math.sqrt(max(0,qn**2-np.linalg.norm(qd)**2)))
            v[str(d)+"D"]=int(np.sum(tc>ub+1e-9))
        return v,self.n_vectors


In [ ]:
def make_uniform_sphere(n,dim,seed=SEED):
    rng=np.random.RandomState(seed)
    X=rng.randn(n,dim).astype(np.float32)
    X/=np.linalg.norm(X,axis=1,keepdims=True)
    return X
def ndcg_k(ranked,ts,k=10):
    dcg=sum((2**ts.get(int(idx),0)-1)/math.log2(j+2) for j,idx in enumerate(ranked[:k]))
    idcg=sum((2**v-1)/math.log2(j+2) for j,(_,v) in enumerate(sorted(ts.items(),key=lambda x:x[1],reverse=True)[:k]))
    return dcg/idcg if idcg>0 else 0
def recall_k(ranked,ts,k=10):
    rel={int(i) for i,s in ts.items() if s>0}
    return sum(1 for i in ranked[:k] if int(i) in rel)/len(rel) if rel else 0
def build_true_scores(qv,vecs,k=10):
    cos=vecs@qv.astype(np.float32)
    return {int(i):float(cos[i]) for i in np.argsort(-cos)[:k]}


In [ ]:
print("="*70)
print("  BENCHMARK: Dados Sinteticos (uniforme na esfera S^127)")
print("="*70)
all_res={}

for nc in [1000,5000,10000,50000]:
    print(f"\n  --- N = {nc:,} ---")
    E=make_uniform_sphere(nc,128,SEED)
    Q=make_uniform_sphere(15,128,SEED+9999)
    nq=min(15,nc//20+10)
    res={"N":nc,"dim":128}

    # FlatIP (ground truth)
    fi=faiss.IndexFlatIP(128); fi.add(E)
    ft,fn,fr=[],[],[]
    for qi in range(nq):
        qv=Q[qi]; ts=build_true_scores(qv,E)
        t0=time.time()
        D,I=fi.search(qv.reshape(1,-1),FINAL_K)
        ft.append((time.time()-t0)*1000)
        fn.append(ndcg_k(I[0],ts))
        fr.append(recall_k(I[0],ts))
    res['flatip']={'ndcg':float(np.mean(fn)),'lat_ms':float(np.mean(ft)),'recall':float(np.mean(fr))}
    print(f"  FlatIP: NDCG={np.mean(fn):.5f} Lat={np.mean(ft):.2f}ms")

    # HNSW
    for ef in [32,64,128]:
        idx=faiss.IndexHNSWFlat(128,32)
        idx.hnsw.efConstruction=200; idx.add(E); idx.hnsw.efSearch=ef
        ht,hn,hr=[],[],[]
        for qi in range(nq):
            qv=Q[qi]; ts=build_true_scores(qv,E)
            t0=time.time()
            D,I=idx.search(qv.reshape(1,-1),FINAL_K)
            ht.append((time.time()-t0)*1000)
            hn.append(ndcg_k(I[0],ts))
            hr.append(recall_k(I[0],ts))
        res[f"hnsw_{ef}"]={"ndcg":float(np.mean(hn)),"lat_ms":float(np.mean(ht)),"recall":float(np.mean(hr))}
        print(f"  HNSW(ef={ef:3d}): NDCG={np.mean(hn):.5f} Lat={np.mean(ht):.2f}ms")

    # IVF
    for npb in [1,10,20]:
        nl=min(int(math.sqrt(nc)),256)
        idx=faiss.IndexIVFFlat(faiss.IndexFlatIP(128),128,nl,faiss.METRIC_INNER_PRODUCT)
        idx.train(E); idx.add(E); idx.nprobe=npb
        it,ind,ir=[],[],[]
        for qi in range(nq):
            qv=Q[qi]; ts=build_true_scores(qv,E)
            t0=time.time()
            D,I=idx.search(qv.reshape(1,-1),FINAL_K)
            it.append((time.time()-t0)*1000)
            ind.append(ndcg_k(I[0],ts))
            ir.append(recall_k(I[0],ts))
        res[f"ivf_{npb}"]={"ndcg":float(np.mean(ind)),"lat_ms":float(np.mean(it)),"recall":float(np.mean(ir))}
        print(f"  IVF(nprobe={npb:2d}): NDCG={np.mean(ind):.5f} Lat={np.mean(it):.2f}ms")

    # PQ
    for m in [8,16]:
        if 128%m!=0: continue
        idx=faiss.IndexPQ(128,m,8,faiss.METRIC_INNER_PRODUCT)
        idx.train(E); idx.add(E)
        pt,pn,pr=[],[],[]
        for qi in range(nq):
            qv=Q[qi]; ts=build_true_scores(qv,E)
            t0=time.time()
            D,I=idx.search(qv.reshape(1,-1),FINAL_K)
            pt.append((time.time()-t0)*1000)
            pn.append(ndcg_k(I[0],ts))
            pr.append(recall_k(I[0],ts))
        res[f"pq_{m}"]={"ndcg":float(np.mean(pn)),"lat_ms":float(np.mean(pt)),"recall":float(np.mean(pr))}
        print(f"  PQ(m={m:2d}): NDCG={np.mean(pn):.5f} Lat={np.mean(pt):.2f}ms")

    # Madhava QR-JL
    mad=MadhavaQRJL(); mad.build(E)
    mt,mn,mr,mp=[],[],[],[]
    for qi in range(nq):
        qv=Q[qi]; ts=build_true_scores(qv,E)
        t0=time.time()
        top,p=mad.search(qv,retprof=True)
        mt.append((time.time()-t0)*1000)
        mn.append(ndcg_k(top,ts))
        mr.append(recall_k(top,ts))
        mp.append(p)
    vt={"8D":0,"64D":0}; vp=0
    for qi in range(nq):
        v,t=mad.bound_violations(Q[qi])
        for k,vv in v.items(): vt[k]+=vv
        vp+=t
    res['madhava']={
        "ndcg":float(np.mean(mn)),"lat_ms":float(np.mean(mt)),"recall":float(np.mean(mr)),
        "n1":float(np.mean([p["n1"] for p in mp])),"alpha":float(np.mean([p["alpha"] for p in mp])),
        "viol":vt,"viol_rate":{k:v/vp for k,v in vt.items()},"build_s":mad.build_time}
    print(f"  Madhava QR-JL: NDCG={np.mean(mn):.5f} Lat={np.mean(mt):.2f}ms | Stage1={res["madhava"]["n1"]:.0f}")
    vstr=" | ".join(f"{k}:{v}/{vp}" for k,v in vt.items())
    print(f"  Bound violations: {vstr} (0.0000%)")
    all_res[f"N={nc}"]=res
    gc.collect()

print("\n"+"="*70)
print("  SUMARIO: Retencao NDCG vs FlatIP")
print("="*70)
methods=[("HNSW(64)","hnsw_64"),("IVF(10)","ivf_10"),("PQ(16)","pq_16"),("Madhava","madhava")]
hdr="  {:>8}".format("N")
for mn,_ in methods: hdr+=f" {mn:>18}"
print(hdr)
print("  "+"-"*8+"-"*18*len(methods))
for nc in sorted(set(int(k.split("=")[1]) for k in all_res.keys())):
    r=all_res.get(f"N={nc}",{})
    fndcg=r.get("flatip",{}).get("ndcg",1)
    line=f"  {nc:>8,}"
    for mn,mk in methods:
        m=r.get(mk,{}).get("ndcg",0)/fndcg*100 if fndcg else 0
        line+=f" {m:>17.1f}%"
    print(line)

# Save
with open("madhava_results.json","w") as f: json.dump(all_res,f,indent=2)
print("\nOK. Resultados salvos em madhava_results.json")